# Data Preparation: corpus de jugadas elegibles

Este notebook procesa todos los archivos locales de eventos de StatsBomb, con independencia de la disponibilidad de datos 360. Sigue las reglas de continuidad de `docs/specs/clustering-de-jugadas.md` y guarda cada jugada elegible en una carpeta de competición, temporada y partido.

La salida incluye un manifiesto por partido y `resumen_corpus.json` con totales, parámetros y errores. Una carpeta de partido existente se conserva salvo que `SOBRESCRIBIR` se active explícitamente.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

from src.data_preparation.corpus import descubrir_archivos_eventos, procesar_corpus
from src.data_preparation.export import leer_jugada

# Configuración reproducible del lote
RUTA_ENTRADA = Path("datos_statsbomb")
RUTA_SALIDA = Path("datos_preparados_corpus")
MIN_PASES_COMPLETADOS = 3
MAX_SEGUNDOS_INTERVENCION = 3
MAX_ACCIONES_INTERVENCION = 3
SOBRESCRIBIR = False
TRABAJADORES = 6


## 1. Descubrimiento del corpus local

In [ ]:
archivos_eventos = descubrir_archivos_eventos(RUTA_ENTRADA)
print(f"Partidos con eventos descubiertos: {len(archivos_eventos)}")
print(f"Primer archivo: {archivos_eventos[0] if archivos_eventos else 'ninguno'}")


## 2. Procesamiento independiente de cada partido

In [ ]:
resumen = procesar_corpus(
    RUTA_ENTRADA,
    RUTA_SALIDA,
    sobrescribir=SOBRESCRIBIR,
    trabajadores=TRABAJADORES,
    min_pases=MIN_PASES_COMPLETADOS,
    max_segundos_intervencion=MAX_SEGUNDOS_INTERVENCION,
    max_acciones_intervencion=MAX_ACCIONES_INTERVENCION,
)
print(f"Estado: {resumen['estado']}")
print(f"Partidos descubiertos: {resumen['partidos_descubiertos']}")
print(f"Partidos procesados: {resumen['partidos_procesados']}")
print(f"Partidos fallidos: {resumen['partidos_fallidos']}")
print(f"Jugadas elegibles guardadas: {resumen['total_jugadas_elegibles']}")
print(f"Secuencias descartadas por pases: {resumen['total_candidatas_descartadas']}")
print(f"Eventos no exportados: {resumen['total_eventos_no_exportados']}")
print(f"Errores por partido: {len(resumen['errores'])}")
print(f"Ruta de salida: {RUTA_SALIDA}")


## 3. Auditoría del lote

In [ ]:
print("Conciliación de partidos:", resumen['partidos_procesados'] + resumen['partidos_fallidos'] == resumen['partidos_descubiertos'])
print("Conciliación de eventos procesados:", resumen['total_eventos_exportados'] + resumen['total_eventos_no_exportados'] == resumen['total_eventos_entrada'])
if resumen['errores']:
    display(pd.DataFrame(resumen['errores']))
if resumen['partidos']:
    display(pd.DataFrame(resumen['partidos']).head(10))


## 4. Verificación de una jugada guardada

In [ ]:
partido_con_jugadas = next((p for p in resumen['partidos'] if p['total_jugadas_elegibles'] > 0), None)
if partido_con_jugadas:
    ruta_manifiesto = Path(partido_con_jugadas['manifiesto'])
    manifiesto = json.loads(ruta_manifiesto.read_text(encoding='utf-8'))
    primera = manifiesto['jugadas'][0]
    jugada = leer_jugada(ruta_manifiesto.parent / primera['archivo'])
    print(f"Jugada: {primera['play_id']}")
    print(f"Eventos: {len(jugada)}; columnas conservadas: {list(jugada.columns) == manifiesto['columnas']}")
    display(jugada[[c for c in ['id', 'period', 'index', 'timestamp', 'type', 'team'] if c in jugada]].head(10))
else:
    print("No hubo jugadas elegibles entre los partidos procesados.")


## 5. Resultado de la ejecución

In [ ]:
print(f"Resumen persistido: {RUTA_SALIDA / 'resumen_corpus.json'}")
if resumen['estado'] != 'completo':
    raise RuntimeError("La ejecución del corpus quedó incompleta. Revise resumen_corpus.json y los errores por partido.")
